In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [3]:
data = pd.read_csv('churn.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## Pre-processing
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [5]:
## Encode categorical variables = Geography and Gender
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [6]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder(sparse_output = False)

one_hot_encoder_geo = one_hot_encoder.fit_transform(data[['Geography']])

one_hot_encoder_geo

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [7]:
one_hot_encoder.get_feature_names_out(['Geography']) ## the same thing will be used while prediction

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
geo_one_hot_encoded_columns = pd.DataFrame(one_hot_encoder_geo, columns=['Geography_France', 'Geography_Germany', 'Geography_Spain'])

In [9]:
geo_one_hot_encoded_columns

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
data = pd.concat([data.drop(['Geography'], axis=1), geo_one_hot_encoded_columns], axis=1)

In [11]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [12]:
## Save encoders
with open('label_encoder_gender.pkl', 'wb') as file:
  pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder.pkl', 'wb') as file:
  pickle.dump(one_hot_encoder, file)

In [13]:
## divide data into ind. and dep. features
X = data.drop(['EstimatedSalary'], axis=1)
y = data['EstimatedSalary']

In [14]:
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,1,0.0,1.0,0.0


In [15]:
y

0       101348.88
1       112542.58
2       113931.57
3        93826.63
4        79084.10
          ...    
9995     96270.64
9996    101699.77
9997     42085.58
9998     92888.52
9999     38190.78
Name: EstimatedSalary, Length: 10000, dtype: float64

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size = 0.20, random_state = 42
)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
pd.DataFrame(X_train) ## 80% of data after scaling

,0,1,2,3,4,5,6,7,8,9,10,11
0,0.356500,0.913248,-0.655786,0.345680,-1.218471,0.808436,0.649203,0.974817,-0.50858,1.001501,-0.579467,-0.576388
1,-0.203898,0.913248,0.294938,-0.348369,0.696838,0.808436,0.649203,0.974817,-0.50858,-0.998501,1.725723,-0.576388
2,-0.961472,0.913248,-1.416365,-0.695393,0.618629,-0.916688,0.649203,-1.025834,1.96626,-0.998501,-0.579467,1.734942
3,-0.940717,-1.094993,-1.131148,1.386753,0.953212,-0.916688,0.649203,-1.025834,1.96626,1.001501,-0.579467,-0.576388
4,-1.397337,0.913248,1.625953,1.386753,1.057449,-0.916688,-1.540351,-1.025834,1.96626,1.001501,-0.579467,-0.576388
...,...,...,...,...,...,...,...,...,...,...,...,...
7995,1.207474,0.913248,1.435808,1.039728,-0.102301,-0.916688,0.649203,0.974817,-0.50858,1.001501,-0.579467,-0.576388
7996,0.314989,-1.094993,1.816097,-1.389442,-1.218471,-0.916688,0.649203,0.974817,-0.50858,1.001501,-0.579467,-0.576388
7997,0.865009,-1.094993,-0.085351,-1.389442,-1.218471,2.533560,-1.540351,-1.025834,1.96626,1.001501,-0.579467,-0.576388
7998,0.159323,0.913248,0.390011,1.039728,1.827259,-0.916688,0.649203,-1.025834,1.96626,1.001501,-0.579467,-0.576388


In [19]:
with open('scaler.pkl', 'wb') as file:
  pickle.dump(scaler, file)

# ANN Regression:

In [20]:
import tensorflow as tf

from tensorflow.keras.models import Sequential  # Sequential Model
from tensorflow.keras.layers import Dense       # Create a neuron or node

In [21]:
number_of_input_nodes = X_train.shape[1]  ## 12

In [22]:
## Build our ANN model
model = Sequential([
       Dense(64, activation= 'relu', input_shape= (number_of_input_nodes, )), ## HL 1 connected with I/P layer with 64 nodes and input of 12 nodes
       Dense(32, activation= 'relu'), ## HL 2 with 32 nodes
       Dense(1) ## O/P layer with BY DEFAULT 'linear' ACTIVATION FUCNTION for REGRESSION
  ]
)

In [23]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
opt = tf.keras.optimizers.Adam(learning_rate = 0.01)

loss = tf.keras.losses.MeanAbsoluteError()

In [25]:
## compile the model
model.compile(
  optimizer = opt,
  loss = loss, ## regression MAE
  metrics = ['mae'] ## MAE
)

In [26]:
## Set up the Tensorboard

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard ## to visualize all the logs
import datetime

log_dir = "regression_logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tf_callback = TensorBoard(log_dir= log_dir, histogram_freq=1)

In [27]:
## Set up Early Stopping 
early_stopping_callback = EarlyStopping(
  monitor='val_loss', ## validation loss
  patience=10, ## wait for 10 epochs
  restore_best_weights=True)

In [35]:
## Training the model
history = model.fit(
  X_train, y_train,
  validation_data = (X_test, y_test),
  epochs=100,
  callbacks=[tf_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 4ms/step - loss: 49497.5469 - mae: 49497.5469 - val_loss: 50286.3242 - val_mae: 50286.3242
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 49490.3672 - mae: 49490.3672 - val_loss: 50282.1172 - val_mae: 50282.1172
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 49460.9844 - mae: 49460.9844 - val_loss: 50308.1836 - val_mae: 50308.1836
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 49416.9922 - mae: 49416.9922 - val_loss: 50328.8555 - val_mae: 50328.8555
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 49363.1758 - mae: 49363.1758 - val_loss: 50450.0078 - val_mae: 50450.0078
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 49383.1406 - mae: 49383.1406 - val_loss: 50400.0352 - val_mae: 50400.0352
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 49340.9141 - mae: 49340.9141 - 

In [36]:
model.save('model.h5')

d:\DATA SCIENCE ML AI\DEEP_LEARNING\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [37]:
## Load Tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [38]:
%tensorboard --logdir regression_logs/fit

Reusing TensorBoard on port 6006 (pid 18772), started 0:00:53 ago. (Use '!kill 18772' to kill it.)

In [39]:
## Evalute on test data
test_loss, test_mae = model.evaluate(X_test, y_test)

63/63 [==============================] - 0s 2ms/step - loss: 50282.1172 - mae: 50282.1172
